# PhysAR-IDS -- NB4 (v6): Attack 2 -- black-box PPO evasion agent -- two variants, 5 seeds

**Target:** the Optuna-tuned XGBoost from NB2 v6, refitted per seed (same GPU backend as NB2/NB3).
**Variants (both are run, per the plan):**

| Variant | Perturbation allowance | What it answers |
|---|---|---|
| **PPO-unconstrained** | each step clipped to +/-0.3 sigma; displacement may accumulate over 10 steps (ceiling 3.0 sigma per feature) | PPO's ceiling with a free hand (the original reference design) |
| **PPO-budget-matched** | the same per-step clip, but **cumulative** displacement from the original sample clamped to +/-0.1 sigma | like-for-like with Attack 1's PGD eps = 0.1: isolates attack *method* from attack *budget* |

## Full specification (reviewer: "PPO was not specified")

| Item | Setting |
|---|---|
| Attacker knowledge | **Black-box, hard-label oracle**: the agent sees only the predicted class after each query (no scores, no gradients, no model parameters). At most 10 queries per sample (+1 initial query) |
| Controlled features | the **60 cyber features** (`CYBER_IDX`: 10 legacy 1 s statistics + 50 causal 2 s / 5 s window statistics). Physical and physics-derived features are never modified |
| State | the 93 current features, the last applied perturbation (60), the last predicted class (one-hot, 6), normalised step (1) = 160 |
| Action | continuous vector in [-0.3, 0.3]^60 (standardised units) |
| Reward | +10 on evasion (prediction = benign) minus 0.01 * ||a||^2; -1 on timeout; -0.01 * ||a||^2 per other step |
| Episode | one attack-class training sample, up to 10 steps, terminates on evasion |
| Algorithm | `sb3_contrib.RecurrentPPO`, `MlpLstmPolicy`, LSTM 1 layer x 64, `n_steps=128`, `batch_size=64`, `lr=3e-4`, `gamma=0.99`, **50,000 timesteps**, one environment |
| Training pool | the 419,101 attack-class **training** rows (the agent queries the *trained* target, never the test set) |
| Evaluation | **all 81,718 attack-class test rows**, deterministic policy, episodes run in batches (identical to the one-at-a-time loop; an equivalence check on 200 rows is printed) |
| Seeds | 42, 7, 123, 2024, 99: a **new target model, new PPO agent (both variants) and new random baseline for every seed** |

## Additions relative to v4

* **Two non-learning baselines at the same budget** (random perturbation, same 10 queries). If PPO does not beat these, "learned adaptive evasion" is not supported.
* **PGD restricted to the same 60 cyber features** at eps = 0.1 (surrogate transfer, from the saved NB3 surrogates) as the like-for-like gradient comparator; NB3's all-feature PGD is not comparable because PPO cannot touch the physical features.
* All 5 seeds on the **full** test set (v4 re-trained 5 agents but scored them on a 1,500-row subsample).
* Per-class evasion with mean +/- std, 95% t-intervals, paired t-tests across seeds (Holm-adjusted over the 5 classes), burst-clustered bootstrap.
* Query cost, clean-miss rate per class, and a **range-plausibility check** (how far adversarial cyber values go beyond the training min/max; many rows sit exactly on a boundary, so 'any value outside' would be uninformative). (This is a range check only; it does not prove that a real network flow could produce those feature values.)

**Limitation to state in the dissertation:** the 50 window features are deterministic functions of the 1 s traffic, so independently editing all 60 columns is more freedom than a real packet-level attacker has.

**Environment (run once in a terminal, not in a notebook cell):** `pip install stable-baselines3 sb3-contrib gymnasium --no-deps` (installing inside Jupyter can reinstall torch and fail on Windows).
**Runtime:** 10 PPO trainings (5 seeds x 2 variants, ~50k single-row oracle queries each); the notebook is resumable: finished agents and seeds are detected and skipped.

## 0. Setup

In [ ]:
import os, sys, random, time, json, hashlib, shutil, platform, warnings
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score
import matplotlib; import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
try:
    from sb3_contrib import RecurrentPPO
    from stable_baselines3.common.callbacks import CheckpointCallback
except ImportError as e:
    raise ImportError('Install in a terminal (not in a cell): pip install stable-baselines3 sb3-contrib gymnasium --no-deps') from e
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False

# ---- PPO / environment settings (unchanged from the reference design) ----
EPS_BOUND = 0.3; MAX_STEPS = 10
EVASION_REWARD = 10.0; STEP_PENALTY_SCALE = 0.01; TIMEOUT_PENALTY = -1.0
TOTAL_TIMESTEPS = 50000; CKPT_FREQ = 5000
EPS_TOTAL_BUDGET = 0.1
VARIANTS = {'unconstrained': None, 'budgeted': EPS_TOTAL_BUDGET}
EVAL_CHUNK = 8192

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
BACKEND_NAME = 'cpu(hist)' if _cpu else 'gpu(cuda,hist)'
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__} fit backend: {BACKEND_NAME}')
import sb3_contrib, stable_baselines3
print(f'sb3-contrib {sb3_contrib.__version__} | stable-baselines3 {stable_baselines3.__version__} | gymnasium {gym.__version__}')
print('Working directory:', os.getcwd())

## 1. Load v6 data and the NB2 tuned configuration

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; N_CYBER = len(CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)

atk_tr = y_train != 0; X_atk_train, y_atk_train = X_train[atk_tr], y_train[atk_tr]
atk_te = y_test != 0;  X_atk_test,  y_atk_test  = X_test[atk_te],  y_test[atk_te]
test_burst = pd.factorize(pd.MultiIndex.from_arrays([y_test, np.asarray(data['PLCID_test']), np.asarray(data['timestamp_test'])]))[0][atk_te]
FEAT_LO, FEAT_HI = X_train.min(0), X_train.max(0)
print(f'Features {N_FEAT} | attacker-controlled (cyber) {N_CYBER} | training pool {X_atk_train.shape} | evaluation rows {X_atk_test.shape} in {len(np.unique(test_burst))} attack bursts')
print('Training pool class counts:', np.bincount(y_atk_train)[1:], '| evaluation class counts:', np.bincount(y_atk_test)[1:])
nb2 = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
nb2_tuned = np.array([nb2['results']['tuned'][s]['test_f1'] for s in SEEDS if s in nb2['results']['tuned']])
print(f'NB2 tuned test F1 over seeds: {nb2_tuned.mean():.4f} +/- {nb2_tuned.std(ddof=1):.4f}')

## 2. Oracle, environment, and the batched evaluator

* **Oracle.** The target is fitted on the GPU (as in NB2/NB3), then switched to CPU for query-time inference (`set_params(device='cpu')`): the agent issues ~50,000 *single-row* queries per training run, where GPU launch overhead dominates. The notebook prints the CPU-vs-GPU label agreement on the whole test set.
* **One transition function** (`apply_action`) is shared by the Gym environment and the batched evaluator, so the two cannot drift apart; an equivalence check runs the Gym loop and the batched loop on the same rows.
* A "clean miss" (an attack row the target already calls benign) counts as evaded after the first step, exactly as in the original loop; clean-miss rates are reported per class so they can be subtracted mentally.

In [ ]:
class Oracle:
    """Hard-label black-box oracle over the fitted XGBoost classifier."""
    def __init__(self, clf): self.clf = clf; self.n_queries = 0
    def predict(self, X):
        X = np.ascontiguousarray(X, dtype=np.float32); self.n_queries += len(X)
        return self.clf.predict(X)

def apply_action(x_cur, x_orig, action, eps_bound=EPS_BOUND, eps_total=None):
    """In place on x_cur[:, CYBER_IDX]; returns the applied (post-clamp) perturbation."""
    a = np.clip(action, -eps_bound, eps_bound).astype(np.float32)
    old = x_cur[:, CYBER_IDX].copy(); prop = old + a
    if eps_total is not None:
        o = x_orig[:, CYBER_IDX]; prop = o + np.clip(prop - o, -eps_total, eps_total)
    x_cur[:, CYBER_IDX] = prop
    return (prop - old).astype(np.float32)

def build_obs(x_cur, last_applied, last_pred, step):
    oh = np.zeros((len(x_cur), 6), dtype=np.float32); oh[np.arange(len(x_cur)), last_pred] = 1.0
    return np.concatenate([x_cur.astype(np.float32), last_applied.astype(np.float32), oh, np.full((len(x_cur), 1), step / MAX_STEPS, dtype=np.float32)], 1)

class IDSEvasionEnv(gym.Env):
    metadata = {'render_modes': []}
    def __init__(self, X_pool, y_pool, oracle, eps_bound=EPS_BOUND, max_steps=MAX_STEPS, seed=None, eps_total_budget=None):
        super().__init__()
        self.X_pool, self.y_pool, self.oracle = X_pool, y_pool, oracle
        self.eps_bound, self.max_steps, self.eps_total_budget = eps_bound, max_steps, eps_total_budget
        self.rng = np.random.RandomState(seed)
        self.action_space = spaces.Box(low=-eps_bound, high=eps_bound, shape=(N_CYBER,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(N_FEAT + N_CYBER + 6 + 1,), dtype=np.float32)
    def _obs(self): return build_obs(self.x_current[None], self.last_applied[None], np.array([self.last_pred]), self.step_count)[0]
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = self.rng.randint(0, len(self.X_pool))
        self.x_orig = self.X_pool[idx].copy(); self.x_current = self.x_orig.copy(); self.y_true = self.y_pool[idx]
        self.step_count = 0; self.last_applied = np.zeros(N_CYBER, dtype=np.float32)
        self.last_pred = int(self.oracle.predict(self.x_current[None])[0]); return self._obs(), {}
    def step(self, action):
        applied = apply_action(self.x_current[None], self.x_orig[None], np.asarray(action, dtype=np.float32)[None], self.eps_bound, self.eps_total_budget)[0]
        self.last_applied = applied; self.step_count += 1
        self.last_pred = int(self.oracle.predict(self.x_current[None])[0]); pen = STEP_PENALTY_SCALE * float(np.sum(applied ** 2))
        terminated = truncated = False
        if self.last_pred == 0: reward = EVASION_REWARD - pen; terminated = True
        elif self.step_count >= self.max_steps: reward = TIMEOUT_PENALTY - pen; truncated = True
        else: reward = -pen
        return self._obs(), reward, terminated, truncated, {'evaded': self.last_pred == 0}

def run_batched(policy, X0, oracle, eps_total, mode='ppo', rng=None):
    """Run one episode per row of X0 in parallel (10 steps max). mode: 'ppo' (deterministic policy) or 'random'."""
    N = len(X0); x_cur = X0.copy(); last_applied = np.zeros((N, N_CYBER), dtype=np.float32)
    last_pred = np.concatenate([oracle.predict(X0[i:i + 65536]) for i in range(0, N, 65536)]).astype(np.int64); clean_pred = last_pred.copy()
    active = np.ones(N, bool); evaded = np.zeros(N, bool); steps = np.zeros(N, np.int64)
    if mode == 'ppo': h = np.zeros((1, N, 64), np.float32); c = np.zeros((1, N, 64), np.float32)
    for step in range(MAX_STEPS):
        idx = np.where(active)[0]
        if len(idx) == 0: break
        obs = build_obs(x_cur[idx], last_applied[idx], last_pred[idx], step)
        if mode == 'ppo':
            act = np.zeros((len(idx), N_CYBER), np.float32)
            for s in range(0, len(idx), EVAL_CHUNK):
                sl = idx[s:s + EVAL_CHUNK]
                a, (hn, cn) = policy.predict(obs[s:s + EVAL_CHUNK], state=(h[:, sl], c[:, sl]), episode_start=np.full(len(sl), step == 0), deterministic=True)
                act[s:s + EVAL_CHUNK] = a; h[:, sl] = hn; c[:, sl] = cn
        else:
            act = rng.uniform(-EPS_BOUND, EPS_BOUND, size=(len(idx), N_CYBER)).astype(np.float32)
        sub = x_cur[idx]; applied = apply_action(sub, X0[idx], act, EPS_BOUND, eps_total); x_cur[idx] = sub; last_applied[idx] = applied
        pred = np.concatenate([oracle.predict(sub[i:i + 65536]) for i in range(0, len(sub), 65536)]).astype(np.int64); last_pred[idx] = pred
        ev = pred == 0; evaded[idx[ev]] = True; steps[idx[ev]] = step + 1; active[idx[ev]] = False
    return {'evaded': evaded, 'steps': steps, 'x_final': x_cur, 'pred_final': last_pred, 'clean_pred': clean_pred}

def per_class(vec_bool):
    return np.array([vec_bool[y_atk_test == c].mean() for c in range(1, 6)])
def mean_steps(res):
    return np.array([res['steps'][(y_atk_test == c) & res['evaded']].mean() if ((y_atk_test == c) & res['evaded']).any() else np.nan for c in range(1, 6)])
def summarise(res):
    disp = np.abs(res['x_final'][:, CYBER_IDX] - X_atk_test[:, CYBER_IDX])
    _ex = lambda Z: np.maximum(np.maximum(FEAT_LO[CYBER_IDX] - Z[:, CYBER_IDX], Z[:, CYBER_IDX] - FEAT_HI[CYBER_IDX]), 0).max(1)
    exc = np.maximum(_ex(res['x_final']) - _ex(X_atk_test), 0)    # sigma ADDED beyond what the clean test row already had outside the training min/max
    return {'per_class_evasion': per_class(res['evaded']), 'overall_evasion': float(res['evaded'].mean()), 'mean_steps_to_evade': mean_steps(res),
            'mean_queries': float(1 + np.where(res['evaded'], res['steps'], MAX_STEPS).mean()),
            'attack_f1': float(f1_score(y_atk_test, res['pred_final'], average='weighted')),
            'max_abs_disp_mean': float(disp.max(1).mean()), 'max_abs_disp_max': float(disp.max()), 'mean_excess_beyond_train_range': float(exc.mean()), 'frac_rows_beyond_range_0.5sigma': float((exc > 0.5).mean()),
            'evaded': res['evaded'].copy()}
print('environment + evaluator defined; cyber features controlled:', N_CYBER)

## 3. Training / loading an agent (resumable, fingerprinted)

A checkpoint or finished agent is reused only if its fingerprint matches the current target model, settings and seed; otherwise it is discarded rather than silently resumed.

In [ ]:
def fp_of(*parts): return hashlib.sha256(repr(parts).encode()).hexdigest()[:16]
def model_fp(oracle, X_probe): return hashlib.sha256(np.asarray(oracle.predict(X_probe)).tobytes()).hexdigest()[:16]
def latest_ckpt(d):
    if not os.path.isdir(d): return None
    c = [f for f in os.listdir(d) if f.endswith('_steps.zip')]
    if not c: return None
    best = max(c, key=lambda f: int(f.rsplit('_steps.zip', 1)[0].rsplit('_', 1)[1])); return os.path.join(d, best)

def get_agent(variant, seed, oracle):
    eps_total = VARIANTS[variant]; name = f'ppo_attack2_{variant}_{PIPELINE_VERSION}_seed{seed}'
    fp = fp_of(model_fp(oracle, X_atk_train[:300]), EPS_BOUND, MAX_STEPS, EVASION_REWARD, STEP_PENALTY_SCALE, TIMEOUT_PENALTY, TOTAL_TIMESTEPS, eps_total, seed, N_CYBER)
    if os.path.exists(name + '.zip') and os.path.exists(name + '.fp') and open(name + '.fp').read() == fp:
        print(f'    [{variant}/{seed}] finished agent found (fingerprint matches) -- loaded'); return RecurrentPPO.load(name), 0.0
    ckdir = f'_ckpt_{name}'; os.makedirs(ckdir, exist_ok=True); fpf = os.path.join(ckdir, '_fp.txt')
    env = IDSEvasionEnv(X_atk_train, y_atk_train, oracle, seed=seed, eps_total_budget=eps_total)
    ck = latest_ckpt(ckdir)
    if ck and os.path.exists(fpf) and open(fpf).read() == fp:
        model = RecurrentPPO.load(ck, env=env); remaining = TOTAL_TIMESTEPS - model.num_timesteps
        print(f'    [{variant}/{seed}] resuming from {model.num_timesteps}/{TOTAL_TIMESTEPS}')
    else:
        shutil.rmtree(ckdir, ignore_errors=True); os.makedirs(ckdir); open(fpf, 'w').write(fp)
        model = RecurrentPPO('MlpLstmPolicy', env, n_steps=128, batch_size=64, learning_rate=3e-4, gamma=0.99, seed=seed, verbose=0,
                             policy_kwargs=dict(lstm_hidden_size=64, n_lstm_layers=1), device='cpu'); remaining = TOTAL_TIMESTEPS
    t0 = time.time()
    if remaining > 0:
        model.learn(total_timesteps=remaining, reset_num_timesteps=False, callback=CheckpointCallback(save_freq=CKPT_FREQ, save_path=ckdir, name_prefix='ckpt'), progress_bar=False)
    model.save(name); open(name + '.fp', 'w').write(fp); shutil.rmtree(ckdir, ignore_errors=True)
    return model, time.time() - t0

## 4. Per-seed run

For each seed: fit the tuned target, verify it against NB2, switch it to the CPU oracle, train both PPO agents, then score on the **full** attack-class test set:
PPO-unconstrained, PPO-budget-matched, random perturbation at both budgets (same 10 queries), and surrogate-transfer PGD restricted to the cyber features (eps = 0.1).
The Gym loop vs batched loop equivalence is checked once, on seed 42.

In [ ]:
def pgd_cyber(sur, X, y, eps=0.1, n_iter=10):
    sur.eval(); m = torch.zeros(N_FEAT, device=DEVICE); m[torch.as_tensor(CYBER_IDX, device=DEVICE)] = 1.0; alpha = eps / 4
    yt = torch.from_numpy(y).long().to(DEVICE); X0 = torch.from_numpy(X).float().to(DEVICE); Xa = X0.clone()
    for _ in range(n_iter):
        Xa.requires_grad_(True); g = torch.autograd.grad(F.cross_entropy(sur(Xa), yt), Xa)[0]
        Xa = Xa.detach() + alpha * m * g.sign(); Xa = X0 + torch.clamp(Xa - X0, -eps, eps)
    return Xa.detach().cpu().numpy()

class SurrogateMLP(nn.Module):
    def __init__(self, n_features, n_classes=6, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, hidden), nn.ReLU(), nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, n_classes))
    def forward(self, x): return self.net(x)

METHODS = ['PGD_cyber_eps0.1', 'Random_unconstrained', 'Random_budgeted', 'PPO_unconstrained', 'PPO_budgeted']
if SMOKE_TEST: TOTAL_TIMESTEPS = 1024
R = {}; SEED_INFO = {}; ADV = {}; t_all = time.time()
for seed in SEEDS:
    pf = f'attack2_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = fp_of(best_params, EPS_BOUND, MAX_STEPS, TOTAL_TIMESTEPS, EPS_TOTAL_BUDGET, seed, X_atk_test.shape, float(X_atk_test[:5].sum()))
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp:
            R[seed] = saved['R']; SEED_INFO[seed] = saved['info']; ADV[seed] = saved.get('adv', {}); print(f'seed {seed}: result file found (fingerprint matches) -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}', flush=True)
    clf = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); clf.fit(X_train, y_train, verbose=False)
    p_fit = clf.predict(X_test); f1_fit = f1_score(y_test, p_fit, average='weighted')
    clf.set_params(device='cpu', n_jobs=-1)          # CPU oracle for single-row queries
    oracle = Oracle(clf); p_cpu = clf.predict(X_test); f1_cpu = f1_score(y_test, p_cpu, average='weighted')
    SEED_INFO[seed] = {'clean_f1_fit_backend': float(f1_fit), 'clean_f1_cpu_oracle': float(f1_cpu), 'label_agreement_fit_vs_cpu': float((p_fit == p_cpu).mean()),
                       'clean_miss_per_class': np.array([(p_cpu[y_test == c] == 0).mean() for c in range(1, 6)])}
    print(f'  tuned clean F1: fit-backend {f1_fit:.4f} | CPU oracle {f1_cpu:.4f} | label agreement {SEED_INFO[seed]["label_agreement_fit_vs_cpu"]:.5f} | clean miss/class {np.round(SEED_INFO[seed]["clean_miss_per_class"], 3).tolist()}')
    R[seed] = {}
    # --- PGD comparator (cyber features only) from the saved NB3 surrogate
    sur = SurrogateMLP(N_FEAT).to(DEVICE); sur.load_state_dict(torch.load(f'surrogate_mlp_{PIPELINE_VERSION}_seed{seed}.pt', map_location=DEVICE))
    Xp = pgd_cyber(sur, X_atk_test, y_atk_test); pp = oracle.predict(Xp)
    R[seed]['PGD_cyber_eps0.1'] = summarise({'evaded': pp == 0, 'steps': np.where(pp == 0, 1, 0), 'x_final': Xp, 'pred_final': pp, 'clean_pred': None})
    del sur
    # --- random-perturbation baselines
    for var, eps_total in VARIANTS.items():
        res = run_batched(None, X_atk_test, oracle, eps_total, mode='random', rng=np.random.RandomState(seed + 500)); R[seed][f'Random_{var}'] = summarise(res)
    # --- PPO agents
    for var, eps_total in VARIANTS.items():
        agent, tt = get_agent(var, seed, oracle)
        tq = time.time(); res = run_batched(agent, X_atk_test, oracle, eps_total, mode='ppo'); R[seed][f'PPO_{var}'] = summarise(res)
        print(f'  PPO-{var:<13} trained {tt / 60:.1f} min | scored full test set in {time.time() - tq:.0f}s | overall evasion {R[seed][f"PPO_{var}"]["overall_evasion"]:.4f}', flush=True)
        if seed == SEEDS[0]:
            ADV.setdefault(seed, {})[var] = res['x_final'].astype(np.float32)
            n_chk = 200 if not SMOKE_TEST else 60; env = IDSEvasionEnv(X_atk_test[:n_chk], y_atk_test[:n_chk], oracle, seed=0, eps_total_budget=eps_total); ev_loop = []
            for i in range(n_chk):
                env.X_pool, env.y_pool = X_atk_test[i:i + 1], y_atk_test[i:i + 1]; env.rng = np.random.RandomState(0); obs, _ = env.reset(); st = None; es = np.ones((1,), bool); got = False
                for _ in range(MAX_STEPS):
                    a, st = agent.predict(obs, state=st, episode_start=es, deterministic=True); es = np.zeros((1,), bool); obs, r, te, tr, info = env.step(a)
                    if info['evaded']: got = True; break
                    if te or tr: break
                ev_loop.append(got)
            agree = float((np.array(ev_loop) == res['evaded'][:n_chk]).mean())
            print(f'  equivalence check ({var}): Gym one-at-a-time loop vs batched evaluator agree on {agree:.4f} of {n_chk} rows'); SEED_INFO[seed][f'equiv_{var}'] = agree
            assert agree >= 0.98, 'batched evaluator disagrees with the Gym loop'
    joblib.dump({'fp': cfg_fp, 'R': R[seed], 'info': SEED_INFO[seed], 'adv': ADV.get(seed, {})}, pf)
    print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min | oracle queries {oracle.n_queries:,}', flush=True)
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## 5. Per-class evasion: mean +/- std over 5 seeds, all methods (full attack-class test set)

In [ ]:
T4 = stats.t.ppf(0.975, df=len(SEEDS) - 1)
def arr(method, key): return np.stack([np.asarray(R[s][method][key]) for s in SEEDS])
atk_names = class_names[1:]
tab = []
for m in METHODS:
    a = arr(m, 'per_class_evasion'); o = arr(m, 'overall_evasion')
    row = {'method': m}; row.update({c: f'{a[:, i].mean():.3f} +/- {a[:, i].std(ddof=1):.3f}' for i, c in enumerate(atk_names)})
    row['ALL'] = f'{o.mean():.3f} +/- {o.std(ddof=1):.3f}'; tab.append(row)
cm = np.stack([SEED_INFO[s]['clean_miss_per_class'] for s in SEEDS]).mean(0)
tab.append({'method': '(clean miss, no attack)', **{c: f'{cm[i]:.3f}' for i, c in enumerate(atk_names)}, 'ALL': f'{np.mean([ (SEED_INFO[s]["clean_miss_per_class"]).mean() for s in SEEDS]):.3f}'})
evasion_table = pd.DataFrame(tab); print(evasion_table.to_string(index=False))

print('\nMean steps-to-evade among evaded rows (seed mean) / mean oracle queries per sample incl. the initial one:')
for m in METHODS[1:]:
    print(f'  {m:<22} steps {np.round(np.nanmean(arr(m, "mean_steps_to_evade"), 0), 2).tolist()}  queries {arr(m, "mean_queries").mean():.2f}')
print('\nAttack-class weighted F1 after the attack (clean attack-class F1 not shown; lower = stronger attack):')
for m in METHODS: print(f'  {m:<22} {arr(m, "attack_f1").mean():.4f} +/- {arr(m, "attack_f1").std(ddof=1):.4f}')
print('\nRange plausibility of adversarial cyber values (training min/max):')
for m in METHODS: print(f'  {m:<22} mean max|displacement| {arr(m, "max_abs_disp_mean").mean():.3f} sigma | largest {arr(m, "max_abs_disp_max").max():.3f} | mean added excess beyond the training min/max {arr(m, "mean_excess_beyond_train_range").mean():.3f} sigma | rows pushed >0.5 sigma further out {arr(m, "frac_rows_beyond_range_0.5sigma").mean():.3f}')

## 6. Statistical comparisons

Paired t-tests across the 5 seeds (same target model per seed; n = 5, so Wilcoxon cannot go below p = 0.0625). Per-class p-values are Holm-adjusted across the 5 classes within each comparison.

In [ ]:
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rank, i in enumerate(o): run = max(run, (len(p) - rank) * p[i]); adj[i] = min(1.0, run)
    return adj
COMPARISONS = [('PPO_budgeted', 'PGD_cyber_eps0.1'), ('PPO_budgeted', 'Random_budgeted'), ('PPO_unconstrained', 'Random_unconstrained'), ('PPO_unconstrained', 'PPO_budgeted')]
stat_rows = []; paired = {}
for a_, b_ in COMPARISONS:
    A, B = arr(a_, 'per_class_evasion'), arr(b_, 'per_class_evasion'); D = A - B
    raw = np.array([stats.ttest_rel(A[:, i], B[:, i]).pvalue if D[:, i].std(ddof=1) > 0 else 1.0 for i in range(5)]); adj = holm(raw)
    oa, ob = arr(a_, 'overall_evasion'), arr(b_, 'overall_evasion'); po = stats.ttest_rel(oa, ob).pvalue if (oa - ob).std(ddof=1) > 0 else 1.0
    paired[(a_, b_)] = {'per_class_diff': D.mean(0), 'p_raw': raw, 'p_holm': adj, 'overall_diff': float((oa - ob).mean()), 'overall_p': float(po)}
    for i, c in enumerate(atk_names): stat_rows.append({'A': a_, 'B': b_, 'class': c, 'A-B': f'{D[:, i].mean():+.3f}', 'p_raw': f'{raw[i]:.4f}', 'p_holm': f'{adj[i]:.4f}'})
    stat_rows.append({'A': a_, 'B': b_, 'class': 'ALL', 'A-B': f'{(oa - ob).mean():+.3f}', 'p_raw': f'{po:.4f}', 'p_holm': '-'})
stat_table = pd.DataFrame(stat_rows); print(stat_table.to_string(index=False))

# burst-clustered CI of overall evasion, pooled over seeds (resampling attack bursts)
rng = np.random.RandomState(42); ub = np.unique(test_burst); bi = np.searchsorted(ub, test_burst); nb_ = len(ub); n_atk = np.bincount(bi, minlength=nb_).astype(float)
ci = {}
for m in ['PPO_budgeted', 'PPO_unconstrained', 'PGD_cyber_eps0.1', 'Random_budgeted']:
    idx = rng.randint(0, nb_, size=(2000, nb_)); boots = []
    for s in SEEDS:
        ev = np.bincount(bi, weights=R[s][m]['evaded'].astype(float), minlength=nb_); boots.append(ev[idx].sum(1) / n_atk[idx].sum(1))
    pooled = np.mean(boots, 0); ci[m] = (float(arr(m, 'overall_evasion').mean()), *np.percentile(pooled, [2.5, 97.5]))
print('\nBurst-clustered 95% CI of overall evasion (pooled over seeds):')
for m, (pt, lo, hi) in ci.items(): print(f'  {m:<22} {pt:.4f}  [{lo:.4f}, {hi:.4f}]')
print(f'  (independent units: {nb_} attack bursts for {len(y_atk_test):,} rows)')

## 7. Figure

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8)); w = 0.16; xs = np.arange(5)
for k, m in enumerate(METHODS):
    a = arr(m, 'per_class_evasion'); ax[0].bar(xs + (k - 2) * w, a.mean(0), w, yerr=a.std(0, ddof=1), capsize=2, label=m)
ax[0].set_xticks(xs); ax[0].set_xticklabels(atk_names, rotation=20); ax[0].set_ylabel('evasion rate (attack -> benign)'); ax[0].set_title('Per-class evasion, tuned XGBoost (mean +/- sd, 5 seeds, full test set)'); ax[0].legend(fontsize=7); ax[0].grid(alpha=.3, axis='y')
o = [arr(m, 'overall_evasion') for m in METHODS]; ax[1].bar(range(len(METHODS)), [x.mean() for x in o], yerr=[x.std(ddof=1) for x in o], capsize=3)
for i, x in enumerate(o): ax[1].scatter([i] * len(x), x, c='k', s=10, zorder=3)
ax[1].set_xticks(range(len(METHODS))); ax[1].set_xticklabels(METHODS, rotation=25, ha='right', fontsize=8); ax[1].set_ylabel('overall evasion'); ax[1].set_title('Overall evasion by method (dots = seeds)'); ax[1].grid(alpha=.3, axis='y')
plt.tight_layout(); plt.savefig(f'attack2_evasion_{PIPELINE_VERSION}.png', dpi=150); plt.show()

# Attack 1 context (NB3, all 93 features, surrogate transfer)
try:
    a1 = joblib.load(f'attack1_results_{PIPELINE_VERSION}.pkl')['per_class_evasion_eps01']['PGD|tuned']
    print('\nContext -- NB3 all-feature PGD eps=0.1 per-class evasion (5-seed mean):', dict(zip(atk_names, np.round(a1.mean(0), 3))))
except Exception as e: print('NB3 results not found:', e)

## 8. Save (Defence 1 and later notebooks read these)

In [ ]:
joblib.dump({
    'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'METHODS': METHODS, 'class_names': class_names, 'attack_class_test_labels': y_atk_test,
    'spec': {'algo': 'sb3_contrib.RecurrentPPO MlpLstmPolicy lstm64x1', 'n_steps': 128, 'batch_size': 64, 'lr': 3e-4, 'gamma': 0.99, 'timesteps': TOTAL_TIMESTEPS,
             'eps_bound': EPS_BOUND, 'max_steps': MAX_STEPS, 'reward': [EVASION_REWARD, STEP_PENALTY_SCALE, TIMEOUT_PENALTY], 'eps_total_budget': EPS_TOTAL_BUDGET,
             'controlled_features': int(N_CYBER), 'oracle': 'hard label, CPU inference of GPU-fitted tuned XGBoost', 'sb3_contrib': sb3_contrib.__version__},
    'per_seed': {s: {m: {k: v for k, v in R[s][m].items() if k != 'evaded'} for m in METHODS} for s in SEEDS}, 'seed_info': SEED_INFO,
    'evasion_table': evasion_table, 'stat_table': stat_table, 'paired': {f'{a}|{b}': v for (a, b), v in paired.items()}, 'burst_ci': ci,
    # legacy keys
    'evasion_results': {c + 1: {'total': int((y_atk_test == c + 1).sum()), 'evaded': int(R[42]['PPO_unconstrained']['evaded'][y_atk_test == c + 1].sum())} for c in range(5)},
    'budget_matched_comparison': {atk_names[i]: {'pgd_cyber_evasion': float(arr('PGD_cyber_eps0.1', 'per_class_evasion').mean(0)[i]), 'ppo_unconstrained_evasion': float(arr('PPO_unconstrained', 'per_class_evasion').mean(0)[i]),
                                                  'ppo_budget_matched_evasion': float(arr('PPO_budgeted', 'per_class_evasion').mean(0)[i])} for i in range(5)},
}, f'attack2_results_{PIPELINE_VERSION}.pkl')
for var in VARIANTS: np.save(f'X_adv_ppo_{var}_{PIPELINE_VERSION}.npy', ADV[SEEDS[0]][var]) if SEEDS[0] in ADV and var in ADV[SEEDS[0]] else None
evasion_table.to_csv(f'attack2_evasion_table_{PIPELINE_VERSION}.csv', index=False); stat_table.to_csv(f'attack2_paired_tests_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB4', 'pipeline_version': PIPELINE_VERSION, 'backend': BACKEND_NAME, 'xgboost': xgb.__version__, 'torch': torch.__version__, 'sb3_contrib': sb3_contrib.__version__,
           'python': platform.python_version(), 'seeds': SEEDS, 'timesteps': TOTAL_TIMESTEPS, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb4_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
files = [f'attack2_results_{PIPELINE_VERSION}.pkl', f'attack2_evasion_table_{PIPELINE_VERSION}.csv', f'attack2_paired_tests_{PIPELINE_VERSION}.csv', f'attack2_evasion_{PIPELINE_VERSION}.png', f'nb4_run_manifest_{PIPELINE_VERSION}.json'] \
        + [f'X_adv_ppo_{v}_{PIPELINE_VERSION}.npy' for v in VARIANTS] + [f'ppo_attack2_{v}_{PIPELINE_VERSION}_seed{s}.zip' for v in VARIANTS for s in SEEDS]
for f in files: print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)